# 00 檢查你的 setup

從上到下執行。每個 code cell 最後都應該看到「通過」。若失敗，請依錯誤訊息打開對應安裝指南，補齊後再重跑。

這份 notebook 是課程的主要 readiness gate。它檢查 notebook kernel、Python packages、必要課程資料、course exporter、Prometheus、Grafana Local，以及 workshop 即時 OS metrics 需要的 exporter。


## 1. Course repo path and required files

確認 notebook 正在 `aiops-anomaly-zero-to-hero` repo 內執行，且 labs 需要的資料、exporter、Prometheus 設定檔與起始 notebooks 都存在。


In [ ]:
from pathlib import Path
import os
import platform
import sys


def find_project_root(start: Path) -> Path | None:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "environment.yml").exists() and (candidate / "labs").exists() and (candidate / "infra").exists():
            return candidate
    return None

repo_root = find_project_root(Path.cwd())
if repo_root is None:
    print("FAIL 找不到 aiops-anomaly-zero-to-hero repo。")
    print("請先 cd 到 repo 根目錄，再啟動 JupyterLab 或重新開啟這份 notebook。")
    raise SystemExit("course repo path 檢查失敗")

os.chdir(repo_root)
current_os = platform.system()
prometheus_config = {
    "Darwin": Path("infra/prometheus/prometheus.macos.yml"),
    "Linux": Path("infra/prometheus/prometheus.linux.yml"),
    "Windows": Path("infra/prometheus/prometheus.windows.yml"),
}.get(current_os)

required_paths = [
    Path("environment.yml"),
    Path("data/synthetic/synthetic_rrd_metrics.csv"),
    Path("infra/rrd_exporter.py"),
    Path("infra/prometheus/alerts.yml"),
    Path("labs/self-study/00_observability_stack.ipynb"),
    Path("labs/workshop/00_observability_stack_and_promql.ipynb"),
]
if prometheus_config:
    required_paths.append(prometheus_config)

missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    print("FAIL course files 不完整：")
    for path in missing_paths:
        print("-", path)
    print("請確認你使用的是完整 clone 下來的 aiops-anomaly-zero-to-hero repo。")
    raise SystemExit("course files 檢查失敗")

print("Course repo:", repo_root)
print("Platform:", current_os, platform.release(), platform.machine())
print("Python:", sys.executable)
if prometheus_config:
    print("Expected Prometheus config:", prometheus_config)
else:
    print("提醒：此 OS 沒有預設 Prometheus config，請參考 labs/getting-started/02-install-prometheus.md。")
print("通過：notebook 已在此 course repo 中執行，必要課程檔案存在。")


## 2. Python kernel / packages

確認目前 notebook kernel 使用課程需要的 Python environment，並檢查 `environment.yml` 中的必要 packages。


In [ ]:
import importlib.metadata as metadata
import importlib.util
import os
import platform
import sys

required_packages = {
    "numpy": ("numpy", "1.26"),
    "pandas": ("pandas", "2.0"),
    "scipy": ("scipy", "1.12"),
    "scikit-learn": ("sklearn", "1.4"),
    "matplotlib": ("matplotlib", "3.9"),
    "jupyterlab": ("jupyterlab", "4.3"),
    "nbconvert": ("nbconvert", "7.16"),
    "ipykernel": ("ipykernel", "6.29"),
    "requests": ("requests", "2.31"),
    "pyyaml": ("yaml", "6.0"),
    "pytest": ("pytest", "8.0"),
    "pytest-cov": ("pytest_cov", "5.0"),
    "prometheus-client": ("prometheus_client", "0.20"),
}

if platform.system() == "Darwin":
    required_packages["appnope"] = ("appnope", "0.1")
elif platform.system() == "Windows":
    required_packages["pywin32"] = ("win32api", "306")


def version_tuple(value: str) -> tuple[int, ...]:
    parts = []
    for part in value.replace("-", ".").split("."):
        digits = "".join(ch for ch in part if ch.isdigit())
        if digits:
            parts.append(int(digits))
        elif parts:
            break
    return tuple(parts)

failures = []
if sys.version_info < (3, 12):
    failures.append(f"Python >= 3.12 required; current={platform.python_version()}")
else:
    print(f"OK Python {platform.python_version()}")

conda_env = os.environ.get("CONDA_DEFAULT_ENV")
if conda_env == "aiops-anomaly-zero-to-hero":
    print("OK conda environment: aiops-anomaly-zero-to-hero")
elif conda_env:
    print(f"WARN conda environment is {conda_env!r}; packages are still checked below.")
else:
    print("WARN CONDA_DEFAULT_ENV is not set; packages are still checked below.")

optional_packages = {
    "anthropic": ("anthropic", "0.40", "Lab 07 / Capstone real LLM API; mock mode works without it"),
}

for package, (module_name, minimum) in required_packages.items():
    if importlib.util.find_spec(module_name) is None:
        failures.append(f"missing package: {package}")
        continue
    try:
        version = metadata.version(package)
    except metadata.PackageNotFoundError:
        failures.append(f"installed module but missing package metadata: {package}")
        continue
    if version_tuple(version) < version_tuple(minimum):
        failures.append(f"{package} >= {minimum} required; current={version}")
        continue
    print(f"OK {package} {version}")

if failures:
    print("FAIL Python environment 尚未就緒：")
    for item in failures:
        print("-", item)
    print()
    print("請依作業系統打開安裝指南：")
    print("- macOS: labs/getting-started/01a-setup-macos-python-environment.md")
    print("- Linux: labs/getting-started/01b-setup-linux-python-environment.md")
    print("- Windows: labs/getting-started/01c-setup-windows-python-environment.md")
    raise SystemExit("Python kernel / packages 檢查失敗")

for package, (module_name, minimum, reason) in optional_packages.items():
    if importlib.util.find_spec(module_name) is None:
        print(f"SKIP optional {package}: {reason}")
        continue
    try:
        version = metadata.version(package)
    except metadata.PackageNotFoundError:
        print(f"SKIP optional {package}: module exists but package metadata is missing")
        continue
    if version_tuple(version) < version_tuple(minimum):
        print(f"SKIP optional {package}: {version} installed, >= {minimum} recommended for {reason}")
    else:
        print(f"OK optional {package} {version}: {reason}")

print("通過：Python kernel 與 environment.yml required packages 已就緒。")


## 3. 啟動並檢查 course exporter

這一格會檢查 `http://localhost:8000/metrics`。若 exporter 尚未啟動，notebook 會在背景啟動 `infra/rrd_exporter.py`，並確認 `/metrics` 真的輸出課程用的 `network_rrd_*` metrics。


In [ ]:
import os
import socket
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

EXPORTER_URL = "http://localhost:8000/metrics"
EXPORTER_LOG = Path("outputs/setup/rrd_exporter.log")
REQUIRED_EXPORTER_MARKERS = ["network_rrd_simulated_timestamp", "network_rrd_inoctets"]


def fetch_text(url: str, timeout: float = 2.0) -> tuple[bool, str]:
    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            text = response.read().decode("utf-8", errors="replace")
            return 200 <= response.status < 500, text
    except (urllib.error.URLError, TimeoutError, socket.timeout):
        return False, ""


def exporter_ready() -> bool:
    ok, text = fetch_text(EXPORTER_URL)
    return ok and all(marker in text for marker in REQUIRED_EXPORTER_MARKERS)

if exporter_ready():
    print(f"OK course exporter 已在執行並輸出課程 metrics: {EXPORTER_URL}")
else:
    print("course exporter 尚未就緒，現在由 notebook 背景啟動。")
    EXPORTER_LOG.parent.mkdir(parents=True, exist_ok=True)
    log_file = EXPORTER_LOG.open("a", encoding="utf-8")
    kwargs = {
        "cwd": Path.cwd(),
        "stdout": log_file,
        "stderr": subprocess.STDOUT,
        "env": os.environ.copy(),
    }
    if os.name == "nt":
        kwargs["creationflags"] = subprocess.CREATE_NEW_PROCESS_GROUP
    else:
        kwargs["start_new_session"] = True
    process = subprocess.Popen([sys.executable, "infra/rrd_exporter.py"], **kwargs)

    for _ in range(20):
        time.sleep(1)
        if exporter_ready():
            print(f"OK course exporter 已啟動並輸出課程 metrics: {EXPORTER_URL}")
            print(f"Log: {EXPORTER_LOG}")
            break
    else:
        ok, text = fetch_text(EXPORTER_URL)
        print("FAIL course exporter 無法提供課程 metrics。")
        print("請查看安裝指南: labs/getting-started/02-install-prometheus.md")
        print(f"Log: {EXPORTER_LOG}")
        if ok:
            print("/metrics 有回應，但缺少：")
            for marker in REQUIRED_EXPORTER_MARKERS:
                if marker not in text:
                    print(f"- {marker}")
        if EXPORTER_LOG.exists():
            print()
            print("Log tail:")
            print("".join(EXPORTER_LOG.read_text(encoding="utf-8", errors="replace").splitlines(True)[-20:]))
        raise SystemExit("course exporter 檢查失敗")

print("通過：course exporter 已就緒。")


## 4. Local services 與 Prometheus scrape 狀態

確認 Prometheus、Grafana Local 是否正在執行，並確認 Prometheus 已 scrape 到 course exporter。`node_exporter` / `windows_exporter` 只有 workshop 即時 OS metrics 需要。


In [ ]:
import json
import platform
import socket
import time
import urllib.error
import urllib.parse
import urllib.request


def http_text(url: str, timeout: float = 2.0) -> tuple[bool, str, int | None]:
    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            text = response.read().decode("utf-8", errors="replace")
            return 200 <= response.status < 500, text, response.status
    except (urllib.error.URLError, TimeoutError, socket.timeout):
        return False, "", None


def prometheus_query(query: str, timeout: float = 3.0) -> tuple[bool, list]:
    url = "http://localhost:9090/api/v1/query?" + urllib.parse.urlencode({"query": query})
    ok, text, _ = http_text(url, timeout=timeout)
    if not ok:
        return False, []
    try:
        payload = json.loads(text)
    except json.JSONDecodeError:
        return False, []
    if payload.get("status") != "success":
        return False, []
    return True, payload.get("data", {}).get("result", [])

required_failures = []
optional_messages = []

prom_ready, _, _ = http_text("http://localhost:9090/-/ready")
if prom_ready:
    print("OK Prometheus service: http://localhost:9090")
else:
    print("FAIL Prometheus: 無法連線到 http://localhost:9090/-/ready")
    required_failures.append(("Prometheus service", "labs/getting-started/02-install-prometheus.md"))

if prom_ready:
    rrd_up = False
    metric_seen = False
    for _ in range(8):
        ok, result = prometheus_query('up{job="rrd-exporter"}')
        rrd_up = ok and any(float(item.get("value", [0, "0"])[1]) == 1.0 for item in result)
        ok_metric, metric_result = prometheus_query("network_rrd_simulated_timestamp")
        metric_seen = ok_metric and len(metric_result) > 0
        if rrd_up and metric_seen:
            break
        time.sleep(2)

    if rrd_up:
        print('OK Prometheus scrape target: up{job="rrd-exporter"} == 1')
    else:
        print('FAIL Prometheus 尚未 scrape 到 course exporter: up{job="rrd-exporter"} != 1')
        required_failures.append(("Prometheus rrd-exporter target", "labs/getting-started/02-install-prometheus.md"))

    if metric_seen:
        print("OK Prometheus 已收到 network_rrd_* 課程 metrics")
    else:
        print("FAIL Prometheus 查不到 network_rrd_simulated_timestamp")
        required_failures.append(("Prometheus course metrics", "labs/getting-started/02-install-prometheus.md"))

grafana_ok, grafana_text, _ = http_text("http://localhost:3000/api/health")
if grafana_ok:
    try:
        health = json.loads(grafana_text)
        db_status = health.get("database") or health.get("database_status") or "unknown"
        print(f"OK Grafana Local: http://localhost:3000/api/health database={db_status}")
    except json.JSONDecodeError:
        print("OK Grafana Local: http://localhost:3000/api/health")
else:
    print("FAIL Grafana Local: 無法連線到 http://localhost:3000/api/health")
    required_failures.append(("Grafana Local", "labs/getting-started/03a-install-grafana-local.md"))

system_name = platform.system()
if system_name == "Windows":
    os_exporter_name = "windows_exporter"
    os_exporter_job = "windows-exporter"
    os_exporter_url = "http://localhost:9182/metrics"
else:
    os_exporter_name = "node_exporter"
    os_exporter_job = "node-exporter"
    os_exporter_url = "http://localhost:9100/metrics"

os_exporter_ok, _, _ = http_text(os_exporter_url)
if os_exporter_ok:
    print(f"OK {os_exporter_name}: {os_exporter_url}")
    if prom_ready:
        ok, result = prometheus_query(f'up{{job="{os_exporter_job}"}}')
        exporter_scraped = ok and any(float(item.get("value", [0, "0"])[1]) == 1.0 for item in result)
        if exporter_scraped:
            print(f'OK Prometheus scrape target: up{{job="{os_exporter_job}"}} == 1')
        else:
            optional_messages.append(f"{os_exporter_name} endpoint is up, but Prometheus is not scraping job={os_exporter_job}. See labs/getting-started/04-install-node-exporter.md and 02-install-prometheus.md.")
else:
    optional_messages.append(f"{os_exporter_name} 未啟動。self-study 可略過，workshop 即時 OS metrics 需要。See labs/getting-started/04-install-node-exporter.md.")

if required_failures:
    print()
    print("FAIL 必要服務尚未就緒。請先依下列指南補齊：")
    for name, guide in required_failures:
        print(f"- {name}: {guide}")
    raise SystemExit("Local services 檢查失敗")

print()
print("通過：self-study 必要服務與 Prometheus course metrics 已就緒。")
if optional_messages:
    print("提醒：若要跑 workshop 即時 OS metrics，請再完成：")
    for message in optional_messages:
        print(f"- {message}")
else:
    print("通過：workshop 即時 OS metrics exporter 也已就緒。")


## 5. 可以開始哪一條 lab？

若前面四個 code cell 都通過，且只有 `node_exporter` / `windows_exporter` 被標成 `SKIP`，可以開始：

```text
labs/self-study/00_observability_stack.ipynb
```

若 `node_exporter` 或 `windows_exporter` 也已通過，可以開始 workshop 路線：

```text
labs/workshop/00_observability_stack_and_promql.ipynb
```

若任一必要項目顯示 `FAIL`，請先依 notebook 提示的安裝指南補齊後再開始 lab。
